# Lecture 3: Python fundamentals and Git

QSS 20 · Modern Statistical Computing · Fall 2026

This notebook holds every code cell shown in class, in the order shown, with short notes between them. Run it from top to bottom after class to check what you saw, or use it as a reference while you work on the activity. Chapters 4 and 5 of the lecture notes give the full explanation.

**Part A** builds a word counter for the *Federalist* essays with plain Python: values and types, lists, loops, dictionaries, conditions, and one function. **Part B** uses Git to record and reverse a change to that counter. Part B is a set of terminal commands, not Python, so it appears as text you type into a terminal.

## The question: who wrote the disputed *Federalist* essays?

Hamilton, Madison, and Jay wrote the 85 *Federalist* essays under one pseudonym. Eleven essays stay disputed between Hamilton and Madison. Mosteller and Wallace (1963) asked whether small, repeated word choices can decide. Function words such as *by*, *from*, *upon*, and *whilst* are unconscious habits, so authors do not monitor them the way they monitor topic words.

The corpus is 85 text files in `data/raw/federalist/`, one essay per file. No column called "style" waits inside them. This lecture builds the machinery to make one: a count per essay for a few chosen words, then a rate that does not depend on essay length.

In [ ]:
from pathlib import Path
import re

# The notebook lives in lectures/; the data live one level up in data/.
ROOT = Path("..") if Path("../data").exists() else Path(".")
FILES = sorted((ROOT / "data/raw/federalist").glob("fp*.txt"))
print(len(FILES), "essay files, first:", FILES[0].name)

## Part A: Python, at review speed

### Every value has a type

Running an expression produces a value. Every value has a type, and the type sets which operations are legal. When code misbehaves, ask `type()` first.

In [ ]:
essay = FILES[0].read_text(encoding="utf-8")
len(essay), type(essay).__name__, 7 / 2

Essay 1 is a 10,388-character `str`. Division with `/` always returns a float, even when the numbers divide evenly.

Two reminders from the slides: `=` assigns and `==` compares, and indentation is grammar in Python, not decoration.

### Names point at objects

`=` binds a name to an object. It does not copy the object. A list is mutable, so code can change the object itself, and two names can then watch one object change.

`words[:3]` below builds a new three-item list. `alias = sample` builds nothing new; it attaches a second name to the same list.

In [ ]:
words = re.sub(r"[^a-z]+", " ", essay.lower()).split()
sample = words[:3]
alias = sample
sample

**Question.** After the next cell runs, what does `sample` hold, and how long is it? Decide before you run it.

In [ ]:
alias.append("added")
sample, len(sample)

`sample` grew to four items, though no line touched it by name. `.copy()` builds a second object when you need two independent lists. This aliasing behavior returns in lecture 4 as the pandas copy-versus-view warning.

In [ ]:
separate = sample.copy()
separate.append("only here")
sample, separate

### Watch the obvious counter fail

Counting a word looks like a one-liner. Try it on a sentence small enough to check by eye.

In [ ]:
phrase = "Nearby, by the sea, by design."
phrase.lower().count("by")

Three, but only two words are *by*. The third hides inside *Nearby*. String `.count()` matches character sequences, not words. The text has to become a list of words first.

### From one long string to a list of tokens

Three steps turn text into countable words:

1. Lowercase, so `By` and `by` match.
2. Replace every run of nonletters with a space.
3. Split on whitespace into a list of tokens.

Read the pattern `[^a-z]+` as "one or more characters that are not lowercase letters." Regular expressions get their own lecture later.

In [ ]:
tokens = re.sub(r"[^a-z]+", " ", phrase.lower()).split()
tokens, tokens.count("by")

A list's `.count()` matches whole elements only, so the answer is two.

### A cleaning rule is a measurement choice

The rule drops apostrophes and numbers. It calls anything between spaces a word. Every count downstream inherits those decisions. A programming choice quietly became a measurement choice. There is no neutral rule, only a stated one, which is why the activity asks you to compare three rules and defend one.

### The for loop: one pass per item

A `for` loop runs its indented block once per item. Pair it with an accumulator: start empty, fill it one pass at a time. Each pass here does four things: select a file, read it, transform it, store a result. Every data-acquisition loop in the course has this shape.

In [ ]:
word_totals = []
for path in FILES[:5]:
    text = path.read_text(encoding="utf-8")
    tokens = re.sub(r"[^a-z]+", " ", text.lower()).split()
    word_totals.append(len(tokens))
word_totals

### The totals already carry a warning

Five essays, and their lengths differ by a quarter already. Across all 85, essay 13 has 960 tokens and essay 83 has 5,716. A raw count of any word partly measures essay length. The fix is a denominator: occurrences per 1,000 tokens. Lecture 1 made the same point about rates; the payoff figure at the end of Part A uses it.

In [ ]:
lengths = {}
for path in FILES:
    tokens = re.sub(r"[^a-z]+", " ", path.read_text(encoding="utf-8").lower()).split()
    lengths[path.stem] = len(tokens)
min(lengths.values()), max(lengths.values())

### The dictionary: a labeled count

A dictionary stores key-value pairs, so the label travels with the number. `{}` starts one empty. Assigning to a key adds the pair or replaces it. A list would force you to remember that position 3 means *upon*.

`words` is the cleaned token list for essay 1, so `.count()` matches whole tokens here.

In [ ]:
counts = {}
for word in ["by", "upon"]:
    counts[word] = words.count(word)
counts

### The shortcut: a comprehension

A comprehension is the one-line form of the loop you just wrote. Use it when the body is one expression you can say aloud.

In [ ]:
targets = ["by", "from", "to", "upon", "whilst", "while"]
{word: words.count(word) for word in targets}

Essay 1 leans Hamilton: *upon* six times, *whilst* never. One essay proves nothing. Whether the lean holds across 85 essays is the question.

### When code fails: read the bottom line first

The cell below raises an error on purpose. The last line of the traceback names the error class and the trigger. Climb upward only to find your own line.

In [ ]:
counts["the"]

When a missing key is expected and harmless, ask for a fallback with `.get()`. Reserve it for cases where absence is fine: an error you can see beats a default that flows quietly into results.

In [ ]:
counts.get("the", 0)

### Conditions encode the historical record

`if` / `elif` / `else` runs exactly one branch: the first whose test is true. `{2, 3, 4, 5, 64}` is a set, built for the question "is this value one of these?"

The essay ranges below come from historical scholarship, not from the text. The broad `else` deserves suspicion: any unmatched number becomes Hamilton, including 0 and 99.

In [ ]:
def known_author(number):
    if number in {2, 3, 4, 5, 64}:
        return "Jay"
    elif 49 <= number <= 57 or number in {62, 63, 18, 19, 20}:
        return "Disputed or joint"
    elif 37 <= number <= 48 or number in {10, 14, 58}:
        return "Madison"
    else:
        return "Hamilton"

known_author(10), known_author(51), known_author(1)

### Check the rule with a cheap census

Compute a census every time you encode a classification. It catches a mistyped boundary immediately. The list comprehension below is the list sibling of the dictionary comprehension.

In [ ]:
labels = [known_author(number) for number in range(1, 86)]
{lab: labels.count(lab) for lab in
 ["Hamilton", "Madison", "Jay", "Disputed or joint"]}

Hamilton 51, Madison 15, Jay 5, disputed or joint 14. The four sum to 85.

### Freeze the rule in a function

`def` names the function and its parameters. `targets` has a default, so a caller can omit it. The docstring says what comes back. `return` hands the result to the caller. Only a returned value can be reused; a printed one goes to nobody.

The default is a tuple, not a list, because Python creates a default once and shares it across every call. An immutable default cannot be changed by accident.

In [ ]:
def token_counts(text, targets=("by", "from")):
    """Return whole-word counts of the target words in text."""
    tokens = re.sub(r"[^a-z]+", " ", text.lower()).split()
    return {t: tokens.count(t) for t in targets}

token_counts("By design, from one author; BY another.")

**Question.** Write down the three results the rule promises before you run the next cell. The third case keeps the *Nearby* failure visible: a future edit that slips back to substring counting would change a known answer. The trailing comma in `("by",)` makes a one-element tuple.

In [ ]:
print("ordinary:", token_counts("By, BY, from!"))
print("empty:", token_counts(""))
print("whole words:", token_counts("nearby", targets=("by",)))

### The payoff: one habit, all 85 essays

Loop, function, rate, labels: everything from Part A in one figure. Each record holds the essay number, its author label, and the rate of *upon* per 1,000 tokens.

In [ ]:
rates = []
for path in FILES:
    n = int(path.stem[2:])                       # "fp07" -> 7
    tokens = re.sub(r"[^a-z]+", " ", path.read_text(encoding="utf-8").lower()).split()
    rates.append({"n": n,
                  "author": known_author(n),
                  "upon": tokens.count("upon") / len(tokens) * 1000})
rates[:3]

In [ ]:
import matplotlib.pyplot as plt

style = {"Hamilton": dict(color="#00693e", marker="o"),
         "Madison": dict(color="#3a6ea5", marker="s"),
         "Jay": dict(color="#8c8c8c", marker="^"),
         "Disputed or joint": dict(facecolor="white", edgecolor="black", marker="o")}

fig, ax = plt.subplots(figsize=(9, 4))
for group, spec in style.items():
    xs = [r["n"] for r in rates if r["author"] == group]
    ys = [r["upon"] for r in rates if r["author"] == group]
    ax.scatter(xs, ys, label=group, **spec)
ax.set_xlabel("Essay number")
ax.set_ylabel('"upon" per 1,000 tokens')
ax.legend(frameon=False)
plt.show()

In [ ]:
# The group means behind the figure.
for group in ["Hamilton", "Madison"]:
    values = [r["upon"] for r in rates if r["author"] == group]
    print(f"{group}: mean {sum(values) / len(values):.2f} per 1,000 tokens over {len(values)} essays")

Hamilton reaches for *upon*; Madison almost never. The disputed essays sit with Madison.

The boundary of the claim: one selected word does not establish authorship. Topic, editing, and the choice of which words to count could all matter. The supervised-learning lecture tests many features out of sample on this same corpus.

**Activity, Part 1.** Open `lec03_activity.ipynb`. Compare three cleaning rules, choose one, write the function, and build a six-essay marker table.

## Part B: version control with Git

Everything below is typed in a terminal, not in a Python cell. Do the practice in a scratch folder outside your course clone, for example `~/Desktop/git-practice`, so the practice repository does not sit inside `QSS20_FA26`.

### Why keep a history of changes?

Two weeks from now, your `token_counts` numbers stop matching your draft. Which cleaning decision moved them? The file holds only the last save. A folder of `wordcount_v2.py`, `wordcount_final.py`, and `wordcount_final_ACTUAL.py` is a history that cannot answer a single question.

### A commit is not a save

A commit is a labeled, recoverable snapshot: what changed, why, who, and when. Saving updates a file; committing adds a point you can return to. Git is not GitHub: Git records history on your machine, and GitHub hosts a copy. Committing uploads nothing. Pulling downloads commits that someone else made.

### Four places a change can live

A change sits in one of four places: the working tree (files on disk), the staging area (what the next commit will contain), the local history (commits), and a remote such as GitHub. `git add` moves a change from the working tree to staging, `git commit` from staging to history, `git push` from history to the remote, and `git pull` from the remote back to your files. `git status` reports where your changes currently sit.

### Start a repository, then read status

One-time setup (name, email, default branch name), then a repository:

```bash
git config --global user.name "Your Name"
git config --global user.email "you@dartmouth.edu"
git config --global init.defaultBranch main

mkdir ~/Desktop/git-practice
cd ~/Desktop/git-practice
# put README.md, wordcount.py, and fp01.txt in this folder
git init          # makes the hidden .git folder
git status
```

```text
On branch main
No commits yet
Untracked files:  README.md  wordcount.py  fp01.txt
```

**Untracked** means Git sees the files but will not protect them.

### The everyday loop: add, then commit

```bash
git add README.md wordcount.py fp01.txt   # stage the current state
git commit -m "Add the word counter, an essay, and a README"
git status                                # -> working tree clean
```

`git add` copies the content as it is right now into staging. A clean tree means every tracked file matches the last commit. Commit messages are imperative and describe one coherent change.

**Question.** You edit `wordcount.py`, run `git add wordcount.py`, edit it again, then commit. Does the commit contain the second edit?

No. `git add` staged a snapshot. The second edit lives only on disk, and `git status` lists the file twice: staged (old) and not staged (new).

### Read the short status

```text
A  api_keys.py      # first column = staging area
 M wordcount.py     # second column = working tree
?? results.csv      # ?? = untracked
```

Two columns: left is staged, right is the working tree. `A` added, `M` modified, `D` deleted. The position carries the meaning.

### Two undos, one dangerous flag

```bash
git restore --staged api_keys.py   # unstage; disk untouched
git restore wordcount.py           # DISCARD disk edits, no undo
```

`--staged` moves a change out of staging and is safe. Without it, `git restore` overwrites the file on disk. Run `git diff <file>` first, every time, so you know what you lose.

### Read the history

```bash
git log --oneline
```

```text
9c1f2ab Ignore words shorter than three characters by default
3d4e5f6 Add the word counter, an essay, and a README
```

A hash names an exact state: "Table 2 came from `9c1f2ab`." `HEAD` is where you stand, and `HEAD~1` is one commit earlier.

### The silent failure Git must catch

The second commit above looked harmless. The counter still runs. But *by* went from 14 to 0 in essay 1. The diff, not the commit message, shows why:

```bash
git diff HEAD~1 HEAD        # what did that commit change?
```

```text
+ tokens = [t for t in tokens if len(t) >= 3]
```

The filter dropped every word shorter than three letters. Nothing crashed; the measurement changed. This is the most important failure of the lecture, because it produces a plausible program state and a wrong number.

### Reverse it, on the record

```bash
git revert --no-edit HEAD   # a new commit that reverses the bad one
```

Revert adds a new commit that undoes the bad one. The mistake and the repair both stay in the history. Rerun the counter and "14 by, 11 from" comes back.

### Recover a deleted file

```bash
rm wordcount.py
git status --short     # ' D' = deleted on disk, not staged
git restore wordcount.py
```

Git can only restore what was committed. Commit often. Nothing recovers a file that was never committed.

### Ignore files on purpose

A `.gitignore` file lists patterns that Git keeps out of `git status` and staging:

```text
figures/generated/     # generated: rebuildable from code
data/raw/              # data: large, often not ours to share
api_keys.py            # credentials: never commit
```

A committed secret is a leaked secret: revoke or rotate the key, because deleting the file in a later commit does not unleak it. `.gitignore` cannot untrack a file that is already committed; `git rm --cached <file>` untracks it while keeping it on disk.

**Activity, Part 2.** Return to `lec03_activity.ipynb`. Order three repository snapshots, choose the safe recovery command, and write the repair's commit message.

### The course repository: clone once, pull often

```bash
git clone https://github.com/KengChiChang/QSS20_FA26.git  # once, during setup
cd QSS20_FA26
git pull                     # before every class and problem set
cp problemsets/ps1/ps1_blank.ipynb problemsets/ps1/ps1_mywork.ipynb
```

The course repository is public, so cloning and pulling need no sign-in. A clone is a complete copy of the files and their history. Edit the copy, never the original, so `git pull` never clashes with your work. If a pull ever stops because you edited a tracked file: `git stash`, then `git pull`, then `git stash pop`.

### The everyday loop, end to end

```bash
git status                # what changed?
git diff                  # what exactly changed?
git add wordcount.py
git diff --staged         # what am I about to commit?
git commit -m "Describe one coherent change"
```

Stage files by name, not with `git add -A`. That is how data files and keys get committed. Add `git push` once you have a GitHub remote of your own; chapter 5 covers remotes, branches, and merge conflicts.

### Problem set 1 arrives Thursday

`git pull` brings `problemsets/ps1/` into your clone. Copy the notebook to `ps1_mywork.ipynb` and work in the copy. Restart and run all, render to PDF, and upload both files to Gradescope, the same path as PS0. Git is how the files reach you, not how you submit.

## What to remember

- Check `type()` first. `=` binds names; it does not copy objects.
- A cleaning rule is a measurement choice. Freeze it in a function and check it on cases with known answers.
- A commit is a recoverable snapshot. The diff finds the bad change; `git revert` reverses it on the record.
- Clone once, pull before class, edit a copy. That is the course-file path from today on.

Next lecture: pandas turns the essay records into a real table.